Setting paths & Mount Google Drive

In [ ]:
#Mount Drive + download books from Project Gutenberg
from google.colab import drive
drive.mount('/content/drive')

import os, urllib.request
BASE = '/content/drive/MyDrive/DSA4213_A1'
DATA = f'{BASE}/data'
OUT  = f'{BASE}/outputs'
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)

# Project Gutenberg ebook IDs : choose 2 from the same author and the last one from another writer
GUTENBERG = {
    'little_women':    514,    # author:Louisa May Alcott
    'little_men':      2788,   # author:Louisa May Alcott
    'pride_prejudice': 1342,   # author:Jane Austen
}
FILES = {}
for name, gid in GUTENBERG.items():
    path = f'{DATA}/{name}.txt'
    if not os.path.exists(path):
        urllib.request.urlretrieve(f'https://www.gutenberg.org/cache/epub/{gid}/pg{gid}.txt', path)
    FILES[name] = f'{name}.txt'
    # sanity check: print the title line so you can confirm it's the right book
    with open(path, encoding='utf-8-sig') as f:
        title = next((l.strip() for l in f if l.startswith('Title:')), '??')
    print(f'{name:16s} -> {title}')

Mounted at /content/drive
little_women     -> Title: Little Women
little_men       -> Title: Little Men: Life at Plumfield With Jo's Boys
pride_prejudice  -> Title: Pride and Prejudice


Reproducibility + Checking Hardware

In [ ]:
import random, time, json, pickle, re
from collections import Counter
import numpy as np
import torch

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else 'NO GPU: switch runtime!')
print('PyTorch:', torch.__version__)

Device: cpu | NO GPU — switch runtime!
PyTorch: 2.11.0+cpu


# 1. Data preparation

---
<p>a. Strip the Project Guternburg license header+footer<br>
b. Split <i>Little Women</i> into chapters<br>
c. <b>Chapter-level</b> splitting (every 10th chapter--> test)<br>
d. Normalise the text & tokenise it<br>
e. Build vocab from <b>train</b> only </p>






a. Loading + Stripping header+footer of license

In [ ]:
def load_gutenberg(path):
    with open(path, encoding='utf-8-sig') as f:
        text = f.read()
    start = re.search(r'\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*', text, re.I)
    if start: text = text[start.end():]
    end = re.search(r'\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK', text, re.I)
    if end: text = text[:end.start()]
    return text.strip()

books = {k: load_gutenberg(f'{DATA}/{f}') for k, f in FILES.items()}
for k, t in books.items():
    print(f'{k:16s} {len(t.split()):>8,} words | starts: {t[:80]!r}')

little_women      186,112 words | starts: 'Little Women\n\nby Louisa May Alcott\n\n\nContents\n\n PART 1\n CHAPTER ONE PLAYING PILG'
little_men        104,972 words | starts: "LITTLE MEN: LIFE AT PLUMFIELD WITH JO'S BOYS\n\nBy Louisa May Alcott\n\n\n     TO\n   "
pride_prejudice   127,360 words | starts: '[Illustration:\n\n                             GEORGE ALLEN\n                      '


b. Split *Little Women* into chapters + c. Chapter-level splitting (every 10th chapter--> test)


In [ ]:

lw = books['little_women']
parts = re.split(r'(?m)^\s*CHAPTER\s+[A-Z0-9\-]+\.?.*$', lw)
chapters = [p.strip() for p in parts if len(p.split()) > 500]   # drops ToC lines / front matter

print('Chapters kept:', len(chapters))   # expect ~47 for the full Little Women corpus
for i, ch in enumerate(chapters[:5]):
    print(i, '|', ' '.join(ch.split()[:12]), '...')
if len(chapters) < 10:
    print('Too few chapters and the heading format differs.')

Chapters kept: 47
0 | PLAYING PILGRIMS “Christmas won’t be Christmas without any presents,” grumbled Jo, lying ...
1 | A MERRY CHRISTMAS Jo was the first to wake in the gray ...
2 | THE LAURENCE BOY “Jo! Jo! Where are you?” cried Meg at the ...
3 | BURDENS “Oh, dear, how hard it does seem to take up our ...
4 | BEING NEIGHBORLY “What in the world are you going to do now, ...


d. Normalise the text & tokenise it

In [ ]:
# 1.3 Normalise + tokenise
def normalise(text):
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('_', '')          # Gutenberg italics markers
    return text.lower()

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?|[.,!?;:\"]")
def tokenize(text):
    return TOKEN_RE.findall(normalise(text))

print(tokenize('"Christmas won\u2019t be Christmas without any presents," grumbled Jo.'))

['"', 'christmas', "won't", 'be', 'christmas', 'without', 'any', 'presents', ',', '"', 'grumbled', 'jo', '.']


e. Build vocab from train only

In [ ]:
# Chapter-level train / val / test split (interleaved)
train_ch, val_ch, test_ch = [], [], []
for i, ch in enumerate(chapters):
    if i % 10 == 9:   test_ch.append(ch)
    elif i % 10 == 8: val_ch.append(ch)
    else:             train_ch.append(ch)

train_tok = [t for ch in train_ch for t in tokenize(ch)]
val_tok   = [t for ch in val_ch   for t in tokenize(ch)]
test_tok  = [t for ch in test_ch  for t in tokenize(ch)]
print(f'Chapters  train/val/test: {len(train_ch)}/{len(val_ch)}/{len(test_ch)}')
print(f'Tokens    train/val/test: {len(train_tok):,}/{len(val_tok):,}/{len(test_tok):,}')

# Vocabulary (train only, min_freq = 2)
MIN_FREQ = 2
PAD, UNK = '<pad>', '<unk>'
counts = Counter(train_tok)
itos = [PAD, UNK] + sorted(w for w, c in counts.items() if c >= MIN_FREQ)
stoi = {w: i for i, w in enumerate(itos)}

def encode(tokens):
    return [stoi.get(t, stoi[UNK]) for t in tokens]

train_ids, val_ids, test_ids = encode(train_tok), encode(val_tok), encode(test_tok)
unk = stoi[UNK]
print('Vocab size:', len(itos), f'(unique train words: {len(counts):,})')
for name, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    print(f'<unk> rate {name:5s}: {100*sum(i == unk for i in ids)/len(ids):.2f}%')
print('Top 15 words:', counts.most_common(15))



Chapters  train/val/test: 39/4/4
Tokens    train/val/test: 184,072/23,355/14,905
Vocab size: 5641 (unique train words: 10,013)
<unk> rate train: 2.38%
<unk> rate val  : 4.58%
<unk> rate test : 5.78%
Top 15 words: [(',', 15691), ('.', 7381), ('and', 6682), ('the', 6243), ('"', 5234), ('to', 4199), ('a', 3695), ('of', 2779), ('i', 2619), ('her', 2560), ('it', 2028), ('in', 2025), ('for', 1846), ('she', 1824), ('you', 1773)]


# Saving to Drive

In [ ]:
prep = dict(itos=itos, stoi=stoi, train_ids=train_ids, val_ids=val_ids, test_ids=test_ids,
            train_tok=train_tok, val_tok=val_tok, test_tok=test_tok,
            n_chapters=(len(train_ch), len(val_ch), len(test_ch)), min_freq=MIN_FREQ, seed=SEED)
with open(f'{OUT}/data_prep.pkl', 'wb') as f:
    pickle.dump(prep, f)
print('Saved ->', f'{OUT}/data_prep.pkl')

Saved -> /content/drive/MyDrive/DSA4213_A1/outputs/data_prep.pkl
